# albis tutorial

This notebook demonstrates the app-facing `albis` workflow:

1. generate one synthetic spatial-transcriptomics dataset,
2. inspect the returned `AnnData`,
3. plot aligned and unaligned coordinates,
4. save the result as `.h5ad`.

It uses small settings so the tutorial stays interactive.

## Setup

Install Albis, including plotting support, in your Python environment:

```bash
pip install albis
```

Download this notebook and open it in Jupyter using the same environment.
If Jupyter is not already installed:

```bash
pip install notebook
jupyter notebook
```

If you are already in a notebook, run `%pip install albis` in a cell to
install into the active kernel, then restart the kernel if needed.


In [ ]:
from pathlib import Path

import albis as ab

print("Albis version:", ab.__version__)


## Generate a small bin-level dataset

`generate_data()` returns exactly one `AnnData` object for the requested output type and slice axis.

In [ ]:
adata = ab.generate_data(
    output="bin",
    slice_axis="Z",
    n_cells=10000,
    n_slices=4,
    n_domains=5,
    marker_genes_per_type=20,
    sphere_radius_um=200.0,
    capture_window_um=(300.0, 300.0),
    bin_size_um=30.0,
    seed=2025,
)

adata

## Inspect the returned object

`describe()` gives a compact summary of the app-relevant fields.

In [ ]:
summary = ab.describe(adata)
summary

In [ ]:
adata.obs.head()

In [ ]:
adata.var.head()

## Plot aligned and unaligned coordinates

`plot()` returns a Matplotlib figure. Use `coordinates="aligned"` for canonical tissue coordinates and `coordinates="unaligned"` for the per-slice rigid transforms.

In [ ]:
fig = ab.plot(
    adata,
    view="2d",
    coordinates="aligned",
    color="domain_true",
    point_size=8,
)
fig

In [ ]:
fig = ab.plot(
    adata,
    view="2d",
    coordinates="unaligned",
    color="slice_id",
    point_size=8,
)
fig

## Generate other app outputs

Use the same public API for spot-level or cell-level data. Only the requested output is generated.

In [ ]:
spot_adata = ab.example_data(output="spot", slice_axis="Y", n_cells=200)
ab.describe(spot_adata)

In [ ]:
cell_adata = ab.example_data(output="cell", slice_axis="X", n_cells=200)
ab.describe(cell_adata)

## Matching unaligned perturbations across modalities

By default, each modality draws its own independent per-slice rotation/translation for `spatial_unaligned`, even with identical `base_seed_unaligned` -- so `output="bin"` and `output="spot"` calls above with the same seed still get different unaligned coordinates. Pass `sync_unaligned_seed=True` (and matching `sphere_radius_um`/`slice_axis`/`base_seed_unaligned`/`max_deg`/`max_shift`/`seed` in every call) to make all modalities share the same per-slice transform instead -- useful for comparing how different modalities' own alignment method resolves an identical starting misalignment.

In [ ]:
shared = dict(
    slice_axis="Z",
    sphere_radius_um=200.0,
    n_cells=200,
    n_slices=3,
    base_seed_unaligned=12345,
    sync_unaligned_seed=True,
)
bin_synced = ab.example_data(output="bin", **shared)
spot_synced = ab.example_data(output="spot", **shared)

perturb_bin = bin_synced.uns["rigid_perturb_inplane"]["spatial_unaligned"]["1"]
perturb_spot = spot_synced.uns["rigid_perturb_inplane"]["spatial_unaligned"]["1"]
print("bin  slice 1 shift:", perturb_bin["t"])
print("spot slice 1 shift:", perturb_spot["t"])
assert perturb_bin["t"] == perturb_spot["t"] and perturb_bin["R"] == perturb_spot["R"]

## Watch out: `domain_type_mix` and non-default domain/cell-type counts

`generate_data()` lets you change `n_domains`/`n_cell_types` freely, but if
you do and don't also supply `domain_type_mix`, every domain silently gets
an identical, fully uniform cell-type distribution -- no domain-specific
enrichment at all. This raises a `UserWarning` so it's impossible to miss:


In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    _ = ab.generate_data(output="cell", n_cells=200, n_domains=3, n_cell_types=5)
    domain_mix_warnings = [str(w.message) for w in caught if "domain_type_mix" in str(w.message)]

print(domain_mix_warnings[0] if domain_mix_warnings else "(no warning raised)")


Fix it by supplying your own `(n_domains, n_cell_types)` matrix -- each row
gets renormalized to a probability distribution automatically, so it doesn't
need to sum to exactly 1:


In [ ]:
import numpy as np

my_mix = np.array([
    [0.6, 0.2, 0.1, 0.05, 0.05],   # domain 0: enriched for type 0
    [0.1, 0.6, 0.2, 0.05, 0.05],   # domain 1: enriched for type 1
    [0.2, 0.2, 0.2, 0.2, 0.2],     # domain 2 (core): uniform, on purpose
])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    adata_custom_mix = ab.generate_data(
        output="cell", n_cells=200, n_domains=3, n_cell_types=5, domain_type_mix=my_mix,
    )
    print("warnings raised:", len(caught))


## Save the result

`save()` writes the object with AnnData's `.h5ad` format and returns the output path.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output_path = ab.save(adata, output_dir / "simulation_bins_z.h5ad")
output_path